In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')
CANDIDATE_DIR = Path('/workspace/outputs/candidates')
FEATURE_DIR = Path('/workspace/outputs/features')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

CONVERTED_DIR = Path('/workspace/outputs/converted')
CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 整数変換済みファイルが存在すれば読み込み、なければ変換して保存
# ============================================================
map_path = CONVERTED_DIR / 'id_maps.json'
transactions_path = CONVERTED_DIR / 'transactions.parquet'
customers_path = CONVERTED_DIR / 'customers.parquet'
articles_path = CONVERTED_DIR / 'articles.parquet'

if all(p.exists() for p in [map_path, transactions_path, customers_path, articles_path]):
    print("整数変換済みファイルを読み込み中...")
    
    with open(map_path, 'r') as f:
        maps = json.load(f)
    customer_id_map = maps['customer_id_map']
    customer_id_reverse = {int(v): k for k, v in customer_id_map.items()}
    article_id_map = maps['article_id_map']
    article_id_reverse = {int(v): k for k, v in article_id_map.items()}
    
    transactions = pl.read_parquet(transactions_path)
    customers = pl.read_parquet(customers_path)
    articles = pl.read_parquet(articles_path)
    
    print(f"読み込み完了!")

else:
    print("整数変換済みファイルが存在しないため変換中...")
    
    transactions_raw = pl.read_csv(
        DATA_DIR / 'transactions_train.csv',
        schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
    ).with_columns(pl.col('t_dat').str.to_date())
    
    customers_raw = pl.read_csv(
        DATA_DIR / 'customers.csv',
        schema_overrides={'customer_id': pl.String}
    )
    
    articles_raw = pl.read_csv(
        DATA_DIR / 'articles.csv',
        schema_overrides={'article_id': pl.String}
    )
    
    # 整数変換マップ作成
    print("  マップ作成中...")
    all_customer_ids = pl.concat([
        transactions_raw.select('customer_id'),
        customers_raw.select('customer_id')
    ]).unique().sort('customer_id')['customer_id'].to_list()
    
    all_article_ids = pl.concat([
        transactions_raw.select('article_id'),
        articles_raw.select('article_id')
    ]).unique().sort('article_id')['article_id'].to_list()
    
    customer_id_map = {v: i for i, v in enumerate(all_customer_ids)}
    customer_id_reverse = {i: v for v, i in customer_id_map.items()}
    article_id_map = {v: i for i, v in enumerate(all_article_ids)}
    article_id_reverse = {i: v for v, i in article_id_map.items()}
    
    # マップを保存
    with open(map_path, 'w') as f:
        json.dump({
            'customer_id_map': customer_id_map,
            'article_id_map': article_id_map,
        }, f)
    print(f"  マップ保存完了: {map_path}")
    
    # 整数変換
    print("  transactions変換中...")
    transactions = (
        transactions_raw
        .with_columns([
            pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
            pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32),
        ])
        .drop_nulls()
    )
    transactions.write_parquet(transactions_path)
    print(f"  transactions保存完了: {transactions_path}")
    
    print("  customers変換中...")
    customers = (
        customers_raw
        .with_columns([
            pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
            pl.when(pl.col('age').is_null()).then(-1)
            .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
            .alias('age_bin'),
            pl.col('age').fill_null(-1).cast(pl.Int32),
            pl.col('FN').fill_null(0),
            pl.col('Active').fill_null(0),
            pl.col('club_member_status').fill_null('UNKNOWN'),
            pl.col('fashion_news_frequency').fill_null('NONE'),
        ])
        .with_columns([
            pl.col('club_member_status').replace_strict(
                {'ACTIVE': 1, 'PRE-CREATE': 0, 'LEFT CLUB': -1, 'UNKNOWN': -2}, default=-2
            ).cast(pl.Int32).alias('club_member_status_enc'),
            pl.col('fashion_news_frequency').replace_strict(
                {'Regularly': 2, 'Monthly': 1, 'NONE': 0, 'None': 0}, default=0
            ).cast(pl.Int32).alias('fashion_news_frequency_enc'),
        ])
        .drop_nulls(subset=['customer_id'])
    )
    customers.write_parquet(customers_path)
    print(f"  customers保存完了: {customers_path}")
    
    print("  articles変換中...")
    articles = (
        articles_raw
        .with_columns(
            pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32)
        )
        .drop_nulls(subset=['article_id'])
    )
    articles.write_parquet(articles_path)
    print(f"  articles保存完了: {articles_path}")
    
    print("整数変換完了!")

last_ts = transactions['t_dat'].max()
print(f"\n全データ最終日: {last_ts}")
print(f"customer数: {len(customer_id_map):,}")
print(f"article数: {len(article_id_map):,}")
print(f"transactions: {len(transactions):,}行")

整数変換済みファイルが存在しないため変換中...
  マップ作成中...
  マップ保存完了: /workspace/outputs/converted/id_maps.json
  transactions変換中...
  transactions保存完了: /workspace/outputs/converted/transactions.parquet
  customers変換中...
  customers保存完了: /workspace/outputs/converted/customers.parquet
  articles変換中...
  articles保存完了: /workspace/outputs/converted/articles.parquet
整数変換完了!

全データ最終日: 2020-09-22
customer数: 1,371,980
article数: 105,542
transactions: 31,788,324行


In [ ]:
import polars as pl

try:
    df_test = pl.DataFrame({'a': [1, 2, 3]})
    result = df_test.lazy().select(pl.col('a') * 2).collect(engine='gpu')
    print("Polars GPU: 利用可能!")
except Exception as e:
    print(f"Polars GPU: 利用不可 → {e}")

Polars GPU: 利用可能!


: 

: 

In [ ]:
# ============================================================
# 特徴量作成関数
# ============================================================

def build_user_features(transactions, customers, cutoff):
    """ユーザーベース特徴量（約15個）"""
    print("  ユーザー特徴量作成中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 基本集計
    user_basic = (
        df_train
        .group_by('customer_id')
        .agg([
            pl.len().alias('total_purchases'),
            pl.col('article_id').n_unique().alias('unique_articles'),
            pl.col('t_dat').max().alias('last_purchase_date'),
            pl.col('t_dat').min().alias('first_purchase_date'),
        ])
        .with_columns([
            (pl.lit(cutoff) - pl.col('last_purchase_date')).dt.total_days().alias('days_since_last_purchase'),
            (pl.lit(cutoff) - pl.col('first_purchase_date')).dt.total_days().alias('days_since_first_purchase'),
        ])
    )
    
    # 直近購買回数
    for days in [7, 14, 30, 60]:
        start = cutoff - timedelta(days=days)
        recent = (
            df_train.filter(pl.col('t_dat') >= pl.lit(start))
            .group_by('customer_id')
            .agg(pl.len().alias(f'purchases_last_{days}d'))
        )
        user_basic = user_basic.join(recent, on='customer_id', how='left').with_columns(
            pl.col(f'purchases_last_{days}d').fill_null(0)
        )
    
    # 直近30日のユニーク商品数
    recent30 = (
        df_train.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=30)))
        .group_by('customer_id')
        .agg(pl.col('article_id').n_unique().alias('unique_articles_last_30d'))
    )
    user_basic = user_basic.join(recent30, on='customer_id', how='left').with_columns(
        pl.col('unique_articles_last_30d').fill_null(0)
    )
    
    # 平均購買間隔
    purchase_dates = (
        df_train
        .sort(['customer_id', 't_dat'])
        .group_by('customer_id')
        .agg(pl.col('t_dat').sort().alias('dates'))
    )
    avg_intervals = []
    for row in purchase_dates.iter_rows(named=True):
        dates = row['dates']
        if len(dates) <= 1:
            avg_intervals.append({'customer_id': row['customer_id'], 'avg_purchase_interval': -1.0})
        else:
            diffs = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
            avg_intervals.append({'customer_id': row['customer_id'], 'avg_purchase_interval': sum(diffs)/len(diffs)})
    
    avg_interval_df = pl.DataFrame(avg_intervals).with_columns(
        pl.col('customer_id').cast(pl.Int32)
    )
    user_basic = user_basic.join(avg_interval_df, on='customer_id', how='left')
    
    # 購買頻度（週あたり購買数）
    user_basic = user_basic.with_columns(
        pl.when(pl.col('days_since_first_purchase') > 0)
        .then(pl.col('total_purchases') / (pl.col('days_since_first_purchase') / 7.0))
        .otherwise(0.0)
        .alias('purchase_frequency_per_week')
    )
    
    # リピート率
    repeat_count = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('count'))
        .with_columns(pl.col('count').clip(upper_bound=2).alias('is_repeat'))
        .group_by('customer_id')
        .agg([
            pl.col('is_repeat').sum().alias('repeat_items'),
            pl.len().alias('total_items')
        ])
        .with_columns(
            (pl.col('repeat_items') / pl.col('total_items')).alias('repeat_rate')
        )
        .select(['customer_id', 'repeat_rate'])
    )
    user_basic = user_basic.join(repeat_count, on='customer_id', how='left').with_columns(
        pl.col('repeat_rate').fill_null(0)
    )
    
    # 1回の買い物での平均購買点数（まとめ買い傾向）
    basket_size = (
        df_train.group_by(['customer_id', 't_dat'])
        .agg(pl.len().alias('basket_size'))
        .group_by('customer_id')
        .agg(pl.col('basket_size').mean().alias('avg_basket_size'))
    )
    user_basic = user_basic.join(basket_size, on='customer_id', how='left').with_columns(
        pl.col('avg_basket_size').fill_null(1.0)
    )
    
    # 購買カテゴリ多様性
    cat_diversity = (
        df_train.join(
            articles.select(['article_id', 'product_type_no']), on='article_id', how='left'
        )
        .group_by('customer_id')
        .agg(pl.col('product_type_no').n_unique().alias('category_diversity'))
    )
    user_basic = user_basic.join(cat_diversity, on='customer_id', how='left').with_columns(
        pl.col('category_diversity').fill_null(0)
    )
    
    # customers属性を結合
    user_features = user_basic.join(
        customers.select([
            'customer_id', 'age', 'age_bin', 'FN', 'Active',
            'club_member_status_enc', 'fashion_news_frequency_enc'
        ]),
        on='customer_id', how='left'
    ).with_columns([
        pl.col('age').fill_null(-1),
        pl.col('age_bin').fill_null(-1),
        pl.col('FN').fill_null(0),
        pl.col('Active').fill_null(0),
        pl.col('club_member_status_enc').fill_null(-2),
        pl.col('fashion_news_frequency_enc').fill_null(0),
    ]).drop(['last_purchase_date', 'first_purchase_date'])
    
    print(f"  ユーザー特徴量: {user_features.shape}")
    return user_features


def build_article_features(transactions, articles, cutoff):
    """商品ベース特徴量（約15個）"""
    print("  商品特徴量作成中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 全期間の累計販売数・ユニーク購買者数
    article_basic = (
        df_train.group_by('article_id')
        .agg([
            pl.len().alias('total_sales'),
            pl.col('customer_id').n_unique().alias('unique_buyers'),
            pl.col('t_dat').min().alias('first_sale_date'),
        ])
        .with_columns(
            (pl.lit(cutoff) - pl.col('first_sale_date')).dt.total_days().alias('days_since_first_sale')
        )
        .drop('first_sale_date')
    )
    
    # 直近7/14/30/60日の販売数
    for days in [7, 14, 30, 60]:
        start = cutoff - timedelta(days=days)
        recent = (
            df_train.filter(pl.col('t_dat') >= pl.lit(start))
            .group_by('article_id')
            .agg(pl.len().alias(f'sales_last_{days}d'))
        )
        article_basic = article_basic.join(recent, on='article_id', how='left').with_columns(
            pl.col(f'sales_last_{days}d').fill_null(0)
        )
    
    # トレンド比率
    article_basic = article_basic.with_columns([
        pl.when(pl.col('sales_last_14d') > 0)
        .then(pl.col('sales_last_7d') / pl.col('sales_last_14d'))
        .otherwise(0.0).alias('trend_7_14'),
        pl.when(pl.col('sales_last_30d') > 0)
        .then(pl.col('sales_last_14d') / pl.col('sales_last_30d'))
        .otherwise(0.0).alias('trend_14_30'),
        pl.when(pl.col('sales_last_60d') > 0)
        .then(pl.col('sales_last_30d') / pl.col('sales_last_60d'))
        .otherwise(0.0).alias('trend_30_60'),
    ])
    
    # 商品の再購買率
    article_repeat = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('count'))
        .group_by('article_id')
        .agg([
            pl.col('count').mean().alias('avg_purchase_count_per_buyer'),
            (pl.col('count') > 1).sum().alias('repeat_buyers'),
            pl.len().alias('total_buyers_r')
        ])
        .with_columns(
            (pl.col('repeat_buyers') / pl.col('total_buyers_r')).alias('article_repeat_rate')
        )
        .select(['article_id', 'avg_purchase_count_per_buyer', 'article_repeat_rate'])
    )
    article_basic = article_basic.join(article_repeat, on='article_id', how='left').with_columns([
        pl.col('avg_purchase_count_per_buyer').fill_null(1.0),
        pl.col('article_repeat_rate').fill_null(0.0),
    ])
    
    # 同product_codeのバリアント数
    variant_count = (
        articles.select(['article_id', 'product_code'])
        .group_by('product_code')
        .agg(pl.len().alias('variant_count'))
    )
    article_basic = article_basic.join(
        articles.select(['article_id', 'product_code', 'product_type_no',
                        'garment_group_no', 'index_group_no', 'section_no', 'department_no']),
        on='article_id', how='left'
    ).join(variant_count, on='product_code', how='left').with_columns(
        pl.col('variant_count').fill_null(1)
    ).drop('product_code')
    
    print(f"  商品特徴量: {article_basic.shape}")
    return article_basic


def build_user_article_features(transactions, articles, customers, cutoff, candidates_df):
    """ユーザー×商品特徴量（約20個）"""
    print("  ユーザー×商品特徴量作成中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    result = candidates_df.select(['customer_id', 'article_id', 'candidate_score', 'rank', 'label'])
    
    # 候補ソースフラグ（一括でjoin）
    source_names = ['timedecay', 'repurchase', 'variant', 'also_bought',
                    'item2item_cf', 'bin_popular', 'purchase_interval']
    
    for name in source_names:
        path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
        if not path.exists():
            result = result.with_columns(pl.lit(0).alias(f'is_{name}'))
            continue
        source_df = (
            pl.read_parquet(path)
            .select(['customer_id', 'article_id'])
            .unique()
            .with_columns(pl.lit(1).alias(f'is_{name}'))
        )
        result = result.join(source_df, on=['customer_id', 'article_id'], how='left').with_columns(
            pl.col(f'is_{name}').fill_null(0)
        )
        print(f"    is_{name}: 完了")
    
    # ユーザー×商品のリピート回数
    repeat_count = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('repeat_count'))
    )
    result = result.join(repeat_count, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    
    # 最終購買からの経過日数（article単位）
    last_purchase_article = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('t_dat').max().alias('last_purchase_date'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('last_purchase_date')).dt.total_days().alias('days_since_last_purchase_article')
        )
        .select(['customer_id', 'article_id', 'days_since_last_purchase_article'])
    )
    result = result.join(last_purchase_article, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('days_since_last_purchase_article').fill_null(-1)
    )
    
    # 年齢ビン別購買率
    age_bin_counts = (
        df_train.join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by(['age_bin', 'article_id']).agg(pl.len().alias('age_bin_article_count'))
    )
    age_bin_totals = (
        df_train.join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by('age_bin').agg(pl.len().alias('age_bin_total'))
    )
    age_bin_purchase_rate = (
        age_bin_counts.join(age_bin_totals, on='age_bin', how='left')
        .with_columns(
            (pl.col('age_bin_article_count') / pl.col('age_bin_total')).alias('age_bin_purchase_rate')
        )
        .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
    )
    result = (
        result
        .join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .join(age_bin_purchase_rate, on=['age_bin', 'article_id'], how='left')
        .with_columns(pl.col('age_bin_purchase_rate').fill_null(0.0))
        .drop('age_bin')
    )
    
    # ユーザー×カテゴリ特徴量
    tx_with_cat = df_train.join(
        articles.select(['article_id', 'product_type_no', 'garment_group_no', 'index_group_no']),
        on='article_id', how='left'
    )
    
    for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
        user_cat = (
            tx_with_cat.filter(pl.col(cat_col).is_not_null())
            .group_by(['customer_id', cat_col])
            .agg(pl.len().alias(f'user_{cat_col}_count'))
        )
        user_total = df_train.group_by('customer_id').agg(pl.len().alias('user_total'))
        user_cat_ratio = (
            user_cat.join(user_total, on='customer_id', how='left')
            .with_columns(
                (pl.col(f'user_{cat_col}_count') / pl.col('user_total')).alias(f'user_{cat_col}_ratio')
            )
            .select(['customer_id', cat_col, f'user_{cat_col}_count', f'user_{cat_col}_ratio'])
        )
        art_cat = articles.select(['article_id', cat_col])
        result = (
            result
            .join(art_cat, on='article_id', how='left')
            .join(user_cat_ratio, on=['customer_id', cat_col], how='left')
            .with_columns([
                pl.col(f'user_{cat_col}_count').fill_null(0),
                pl.col(f'user_{cat_col}_ratio').fill_null(0.0),
            ])
            .drop(cat_col)
        )
        print(f"    {cat_col}: 完了")
    
    # ユーザーが同カテゴリを最後に買った日からの経過日数
    for cat_col in ['product_type_no', 'garment_group_no']:
        last_cat_purchase = (
            tx_with_cat.filter(pl.col(cat_col).is_not_null())
            .group_by(['customer_id', cat_col])
            .agg(pl.col('t_dat').max().alias('last_cat_date'))
        )
        art_cat = articles.select(['article_id', cat_col])
        result = (
            result
            .join(art_cat, on='article_id', how='left')
            .join(last_cat_purchase, on=['customer_id', cat_col], how='left')
            .with_columns(
                pl.when(pl.col('last_cat_date').is_not_null())
                .then((pl.lit(cutoff) - pl.col('last_cat_date')).dt.total_days())
                .otherwise(-1)
                .alias(f'days_since_last_{cat_col}')
            )
            .drop([cat_col, 'last_cat_date'])
        )
        print(f"    days_since_last_{cat_col}: 完了")
    
    print(f"  ユーザー×商品特徴量: {result.shape}")
    return result

print("関数定義完了")


def build_all_features(transactions, articles, customers, cutoff, candidates_df):
    """全特徴量を結合"""
    print(f"\n特徴量作成開始（cutoff={cutoff}）...")
    
    # 各特徴量を作成
    user_feat = build_user_features(transactions, customers, cutoff)
    article_feat = build_article_features(transactions, articles, cutoff)
    ua_feat = build_user_article_features(transactions, articles, customers, cutoff, candidates_df)
    
    # 候補DFに結合
    result = ua_feat
    result = result.join(user_feat, on='customer_id', how='left')
    result = result.join(article_feat, on='article_id', how='left')
    
    # null埋め
    for col in result.columns:
        if result[col].dtype in [pl.Float64, pl.Float32]:
            result = result.with_columns(pl.col(col).fill_null(0.0))
        elif result[col].dtype in [pl.Int32, pl.Int64]:
            result = result.with_columns(pl.col(col).fill_null(-1))
    
    print(f"\n特徴量作成完了: {result.shape}")
    print(f"特徴量数: {len(result.columns)}")
    return result

print("特徴量作成関数定義完了")

関数定義完了
特徴量作成関数定義完了


: 

In [ ]:
# ============================================================
# 学習データ作成
# ============================================================
from pathlib import Path

def load_candidates(name, cutoff):
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    if path.exists():
        return pl.read_parquet(path)
    return None

def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

# best_weightsを読み込み
weights_path = CANDIDATE_DIR / f"best_weights_2020-09-15.json"
with open(weights_path, 'r') as f:
    weights_data = json.load(f)
best_weights = weights_data['weights']
print(f"best_weights: {best_weights}")

# 使用するロジック（重み>0のみ）
active_logics = {k: v for k, v in best_weights.items() if v > 0}
print(f"使用ロジック: {list(active_logics.keys())}")

def create_train_dataset(transactions, articles, customers, cutoff, active_logics, n_candidates=100):
    """学習データ作成"""
    print(f"\n学習データ作成中（cutoff={cutoff}）...")
    
    actuals = get_actuals(transactions, cutoff)
    valid_customers = list(actuals.keys())
    print(f"  valid_customers: {len(valid_customers):,}")
    
    # 候補ファイル読み込み
    cand_dfs = {}
    for name in active_logics.keys():
        cached = load_candidates(name, cutoff)
        if cached is None:
            print(f"  {name}: ファイルなし！スキップ")
            continue
        cand_dfs[name] = cached.filter(pl.col('customer_id').is_in(valid_customers))
    
    # スコア正規化・重み付き統合
    scored_list = []
    for name, w in active_logics.items():
        if name not in cand_dfs:
            continue
        df = cand_dfs[name]
        normalized = (
            df.select(['customer_id', 'article_id', 'score'])
            .join(
                df.group_by('customer_id').agg([
                    pl.col('score').max().alias('max_score'),
                    pl.col('score').min().alias('min_score'),
                ]),
                on='customer_id', how='left'
            )
            .with_columns(
                pl.when(pl.col('max_score') == pl.col('min_score'))
                .then(1.0)
                .otherwise(
                    (pl.col('score') - pl.col('min_score')) /
                    (pl.col('max_score') - pl.col('min_score'))
                )
                .alias('norm_score')
            )
            .with_columns((pl.col('norm_score') * w).alias('weighted_score'))
            .select(['customer_id', 'article_id', 'weighted_score'])
        )
        scored_list.append(normalized)
    
    # スコア合算
    combined = pl.concat(scored_list)
    scored_df = (
        combined
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('weighted_score').sum().alias('candidate_score'))
        .with_columns(
            pl.col('candidate_score').rank(method='ordinal', descending=True)
            .over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= n_candidates)
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  候補数: {len(scored_df):,}")
    
    # labelを付与
    actuals_df = pl.DataFrame([
        {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
        for cid, aids in actuals.items()
        for aid in aids
    ]).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
    ])
    
    scored_df = scored_df.join(
        actuals_df, on=['customer_id', 'article_id'], how='left'
    ).with_columns(pl.col('label').fill_null(0))
    
    # 特徴量作成
    features_df = build_all_features(transactions, articles, customers, cutoff, scored_df)
    
    print(f"  完了: {features_df.shape} / 正例: {features_df['label'].sum():,}")
    return features_df

# 学習データ作成（1fold: cutoff=2020-09-08）
TRAIN_CUTOFF = date(2020, 9, 8)
train_df = create_train_dataset(
    transactions, articles, customers, TRAIN_CUTOFF, active_logics
)

# 保存
train_path = FEATURE_DIR / f"train_features_{TRAIN_CUTOFF}.parquet"
train_df.write_parquet(train_path)
print(f"\n保存完了: {train_path}")
print(f"学習データ: {len(train_df):,}行 / 特徴量数: {len(train_df.columns)}")
print(f"正例数: {train_df['label'].sum():,} / 正例率: {train_df['label'].mean():.4f}")

best_weights: {'item2item_cf': 1.0, 'category_popular': 0.0, 'bin_popular': 1.5, 'general_pred': 0.0, 'purchase_interval': 1.0, 'user_category_popular': 0.0, 'timedecay': 3.5, 'repurchase': 1.5, 'variant': 1.0, 'also_bought': 1.0}
使用ロジック: ['item2item_cf', 'bin_popular', 'purchase_interval', 'timedecay', 'repurchase', 'variant', 'also_bought']

学習データ作成中（cutoff=2020-09-08）...
  valid_customers: 72,019
  候補数: 6,574,651

特徴量作成開始（cutoff=2020-09-08）...
  ユーザー特徴量作成中...
  ユーザー特徴量: (1351314, 21)
  商品特徴量作成中...
  商品特徴量: (102967, 19)
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了


In [ ]:
import cudf
print(f"cuDF version: {cudf.__version__}")

df_test = cudf.DataFrame({'a': [1, 2, 3]})
print(df_test * 2)
print("cuDF: 利用可能!")

ModuleNotFoundError: No module named 'numba.cuda.core'